# Build Illustrative Product & Location Dimension Tables

FreshRetailNet anonymizes everything to integers (`product_id`, `city_id`, integer category IDs) — no product names, no regions. To let business users ask natural-language questions over the forecast (and to have orders name a real product), this notebook maps those IDs to friendly labels ("Skim Milk", "Northeast").

These dimension tables are **synthetic and illustrative only** — they are not real data from any customer.


## 1. Configuration & vocabularies

In [ ]:
CATALOG, SCHEMA = "mmf", "fresh_retail_net"
SRC = f"{CATALOG}.{SCHEMA}.demand_train"   # real product_id / city_id live here

# Grocery vocabulary (illustrative). Structure styled after a public synthetic retail dataset
# (category/subcategory/brand/region/store_type), values are grocery to fit FreshRetailNet + the
# 'Skim Milk' narrative.
CATEGORIES = {
    "Dairy":      ["Fresh Milk", "Yogurt", "Cheese", "Butter & Spreads"],
    "Produce":    ["Leafy Greens", "Fresh Fruit", "Root Vegetables", "Herbs"],
    "Bakery":     ["Bread", "Pastries", "Tortillas"],
    "Meat & Seafood": ["Poultry", "Beef", "Pork", "Seafood"],
    "Beverages":  ["Juice", "Bottled Water", "Soft Drinks", "Cold Brew"],
    "Frozen":     ["Frozen Meals", "Ice Cream", "Frozen Vegetables"],
    "Pantry":     ["Cereal", "Pasta & Rice", "Canned Goods", "Snacks"],
}
PRODUCT_WORDS = {
    "Fresh Milk": ["Skim Milk", "Whole Milk", "2% Milk", "Oat Milk", "Almond Milk"],
    "Yogurt": ["Greek Yogurt", "Vanilla Yogurt", "Strawberry Yogurt"],
    "Leafy Greens": ["Baby Spinach", "Romaine Lettuce", "Kale"],
    "Fresh Fruit": ["Bananas", "Strawberries", "Blueberries", "Avocados"],
    "Bread": ["Whole Wheat Bread", "Sourdough Loaf", "Multigrain Bread"],
    "Poultry": ["Chicken Breast", "Ground Turkey"],
    "Juice": ["Orange Juice", "Apple Juice"],
    "Ice Cream": ["Vanilla Ice Cream", "Chocolate Ice Cream"],
    "Cereal": ["Granola", "Corn Flakes", "Oat Cereal"],
}
BRANDS = ["FreshCo", "DailyHarvest", "GreenLeaf", "PureValley", "MarketBasket", "Orchard&Co"]

# Regions / cities (illustrative US). 18 cities -> spread across regions.
REGIONS = ["Northeast", "Southeast", "Midwest", "Southwest", "West", "Northwest"]
CITY_NAMES = ["Boston", "New York", "Philadelphia", "Atlanta", "Miami", "Charlotte",
              "Chicago", "Detroit", "Minneapolis", "Dallas", "Houston", "Phoenix",
              "Los Angeles", "San Francisco", "Seattle", "Portland", "Denver", "Las Vegas"]
REGION_OF_CITY = {  # keep city->region geographically sensible
    "Boston": "Northeast", "New York": "Northeast", "Philadelphia": "Northeast",
    "Atlanta": "Southeast", "Miami": "Southeast", "Charlotte": "Southeast",
    "Chicago": "Midwest", "Detroit": "Midwest", "Minneapolis": "Midwest",
    "Dallas": "Southwest", "Houston": "Southwest", "Phoenix": "Southwest",
    "Los Angeles": "West", "San Francisco": "West", "Las Vegas": "West",
    "Seattle": "Northwest", "Portland": "Northwest", "Denver": "West",
}
STATE_OF_CITY = {
    "Boston": "MA", "New York": "NY", "Philadelphia": "PA", "Atlanta": "GA", "Miami": "FL",
    "Charlotte": "NC", "Chicago": "IL", "Detroit": "MI", "Minneapolis": "MN", "Dallas": "TX",
    "Houston": "TX", "Phoenix": "AZ", "Los Angeles": "CA", "San Francisco": "CA",
    "Seattle": "WA", "Portland": "OR", "Denver": "CO", "Las Vegas": "NV",
}
STORE_TYPES = ["Hypermarket", "Supermarket", "Express"]

# Hero mapping (so the blog's flagship 'Skim Milk in the Northeast' question hits a real surge SKU).
HERO_PRODUCT_ID, HERO_PRODUCT_NAME, HERO_CATEGORY, HERO_SUBCAT = 122, "Skim Milk", "Dairy", "Fresh Milk"
HERO_CITY_ID, HERO_CITY_NAME, HERO_REGION, HERO_STATE = 0, "Boston", "Northeast", "MA"
print("vocab ready")

## 2. product_dim — deterministic synthetic names keyed to real product_id

In [ ]:
from pyspark.sql import functions as F

product_ids = sorted(r.product_id for r in spark.table(SRC).select("product_id").distinct().collect())
city_ids    = sorted(r.city_id    for r in spark.table(SRC).select("city_id").distinct().collect())
print(f"distinct products: {len(product_ids)} | distinct cities: {len(city_ids)}")

cat_list = list(CATEGORIES.items())

def name_for_product(pid: int):
    """Deterministic: pid -> (name, category, subcategory, brand, base_price)."""
    if pid == HERO_PRODUCT_ID:
        return (HERO_PRODUCT_NAME, HERO_CATEGORY, HERO_SUBCAT,
                BRANDS[pid % len(BRANDS)], round(1.99 + (pid % 7) * 0.50, 2))
    cat, subs = cat_list[pid % len(cat_list)]
    sub = subs[(pid // len(cat_list)) % len(subs)]
    words = PRODUCT_WORDS.get(sub, [sub])
    base = words[(pid // 7) % len(words)]
    # make names unique-ish by appending a pack/size variant deterministically
    variant = ["", " (Family Pack)", " (Single)", " 1L", " 2L", " 500g", " 12-pack"][pid % 7]
    return (f"{base}{variant}", cat, sub, BRANDS[pid % len(BRANDS)],
            round(1.49 + (pid % 20) * 0.40, 2))

prod_rows = []
for pid in product_ids:
    nm, cat, sub, brand, price = name_for_product(int(pid))
    prod_rows.append((int(pid), nm, cat, sub, brand, float(price)))

product_dim = spark.createDataFrame(
    prod_rows, ["product_id", "product_name", "category", "subcategory", "brand", "base_price"])
(product_dim.write.mode("overwrite").option("overwriteSchema", "true")
  .saveAsTable(f"{CATALOG}.{SCHEMA}.product_dim"))
print(f"wrote product_dim: {product_dim.count()} rows")
display(product_dim.orderBy("product_id").limit(8))

## 3. location_dim — deterministic synthetic city/region keyed to real city_id

In [ ]:
loc_rows = []
for idx, cid in enumerate(city_ids):
    cid = int(cid)
    if cid == HERO_CITY_ID:
        city, region, state = HERO_CITY_NAME, HERO_REGION, HERO_STATE
    else:
        city = CITY_NAMES[idx % len(CITY_NAMES)]
        region = REGION_OF_CITY.get(city, REGIONS[idx % len(REGIONS)])
        state = STATE_OF_CITY.get(city, "NA")
    store_type = STORE_TYPES[cid % len(STORE_TYPES)]
    loc_rows.append((cid, city, region, state, store_type))

location_dim = spark.createDataFrame(
    loc_rows, ["city_id", "city_name", "region", "state", "store_type"])
(location_dim.write.mode("overwrite").option("overwriteSchema", "true")
  .saveAsTable(f"{CATALOG}.{SCHEMA}.location_dim"))
print(f"wrote location_dim: {location_dim.count()} rows")
display(location_dim.orderBy("city_id"))

## 4. Verify the hero SKU and that dims join to the forecast

Surge SKU `18_122` should resolve to **Skim Milk** in the **Northeast** (Boston).

In [ ]:
%sql
SELECT s.unique_id, p.product_name, p.category, l.city_name, l.region,
       ROUND(SUM(f.forecast_value), 1) AS forecast_7d
FROM mmf.fresh_retail_net.demand_train s
JOIN mmf.fresh_retail_net.product_dim  p ON s.product_id = p.product_id
JOIN mmf.fresh_retail_net.location_dim l ON s.city_id    = l.city_id
JOIN mmf.fresh_retail_net.scoring_output_mv f ON s.unique_id = f.unique_id
WHERE s.unique_id IN ('18_122','299_300','3_191','561_261','450_300','375_4')
GROUP BY s.unique_id, p.product_name, p.category, l.city_name, l.region
ORDER BY forecast_7d DESC;

## 5. Next steps

Add `product_dim` and `location_dim` to the **Fresh Retail Sales Forecasting** Genie space, and extend the
Genie instructions so it maps names → ids:

> A SKU's product name comes from `product_dim` (join on product_id); its region/city from `location_dim`
> (join on city_id). When a user names a product (e.g. "Skim Milk") or region (e.g. "Northeast"), resolve
> it via these tables to the relevant unique_id(s) before querying demand or forecasts.

The supplier feed (`supply_chain.supplier_availability` in S3 Tables) joins on `product_id` + `city_id`, so
orders can display the product name and region while matching on the real ids.